#VERSION 4 AMELIORATION DE VERSION 2

In [ ]:
import nltk
try:
    nltk.data.find('corpora/stopwords')
except nltk.downloader.DownloadError:
    nltk.download('stopwords')
except LookupError:
    nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


In [ ]:
import pandas as pd
import numpy as np
import re
import unicodedata
import nltk
from nltk.stem.snowball import SnowballStemmer

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from xgboost import XGBClassifier

from sklearn.metrics import f1_score, classification_report, confusion_matrix, precision_recall_curve
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline # Pipeline compatible avec SMOTE

import matplotlib.pyplot as plt
import seaborn as sns

# Pour la reproductibilité
SEED = 42
np.random.seed(SEED)

print("--- DÉMARRAGE DE LA VERSION ULTIMATE ---")

# ===================================================
# 1. FONCTIONS DE NETTOYAGE
# ===================================================
def clean_text(text):
    text = str(text).lower()
    text = unicodedata.normalize('NFKD', text).encode('ascii', 'ignore').decode('utf-8')
    text = re.sub(r'\d{1,2}/\d{1,2}/\d{2,4}', ' date ', text) # Remplacement par token générique
    text = re.sub(r'\d+(?:[\.,]\d+)?\s*(?:mg|ml|cp|g|kg)', ' dosage ', text) # Remplacement dosages
    text = re.sub(r'[^a-z\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()

    # Stemming (gardé de la V2)
    stemmer = SnowballStemmer('french')
    tokens = text.split()
    stemmed_tokens = [stemmer.stem(word) for word in tokens if len(word) > 2] # Ignore mots très courts
    return ' '.join(stemmed_tokens)

# ===================================================
# 2. CHARGEMENT ET PRÉPARATION (AMÉLIORATION #4: Feature Engineering)
# ===================================================
print("Chargement des données...")
data = pd.read_csv('/content/data_defi3.csv', sep=';')

# Nettoyage basique des colonnes utiles
required_columns = ['Avis.Pharmaceutique', 'PLT', 'Libellé.Prescription']
for col in required_columns:
    if col not in data.columns:
        print(f"Erreur: La colonne '{col}' est manquante dans le fichier CSV.")
        # Optionnel: arrêter l'exécution ou gérer la colonne manquante différemment
        # Pour l'instant, on va s'arrêter ici si une colonne essentielle manque
        raise KeyError(f"Colonne requise manquante: '{col}'")

data = data.dropna(subset=required_columns)
data['PLT'] = pd.to_numeric(data['PLT'], errors='coerce').fillna(0).astype(int)

# Application du nettoyage de texte
print("Nettoyage du texte...")
data['Avis_Cleaned'] = data['Avis.Pharmaceutique'].apply(clean_text)
# Nettoyage simple de la molécule (minuscule)
data['Molecule_Cleaned'] = data['Libellé.Prescription'].astype(str).str.lower().str.strip()

# Définition des cibles
y_binary = data['PLT'].isin([4, 5, 6.3, 6.4]).astype(int)
y_multi = data['PLT']

# Définition des features (Texte + Molécule)
X = data[['Avis_Cleaned', 'Molecule_Cleaned']]

# Division Train/Test (Stratifiée sur le multiclasse pour garantir la répartition)
print("Division Train/Test...")
X_train, X_test, y_train_bin, y_test_bin, y_train_multi, y_test_multi = train_test_split(
    X, y_binary, y_multi, test_size=0.2, random_state=SEED, stratify=y_multi
)

# ===================================================
# 3. CRÉATION DU PRÉPROCESSEUR (Text + Molécule)
# ===================================================
# Stopwords français enrichis
french_stopwords = nltk.corpus.stopwords.words('french') + \
                  ['patient', 'traitement', 'prescription', 'jour', 'matin', 'soir']

# Définition du ColumnTransformer pour traiter différemment le texte et la molécule
# AMÉLIORATION #1 (partielle) : Paramètres TF-IDF optimisés (ngram 1-3, max_features augmenté)
preprocessor = ColumnTransformer(
    transformers=[
        ('txt', TfidfVectorizer(stop_words=french_stopwords, ngram_range=(1, 3),
                                max_features=5000, min_df=3, sublinear_tf=True), 'Avis_Cleaned'),
        ('mol', OneHotEncoder(handle_unknown='ignore', sparse_output=True), ['Molecule_Cleaned']) # AMÉLIORATION #4
    ],
    remainder='drop'
)

print("--- TÂCHE 1 : CLASSIFICATION BINAIRE (Grave/Non Grave) ---")
# ===================================================
# 4. TÂCHE 1 : Modèles Puissants + Ensembling + SMOTE + Optimisation
# ===================================================

# Définition des modèles de base (AMÉLIORATION #2: Modèles puissants)
clf1 = LogisticRegression(max_iter=2000, solver='liblinear', C=1.0, random_state=SEED)
clf2 = RandomForestClassifier(n_estimators=200, max_depth=None, random_state=SEED, n_jobs=-1)
# Scale_pos_weight aide XGBoost à gérer le déséquilibre
ratio = float(np.sum(y_train_bin == 0)) / np.sum(y_train_bin == 1)
clf3 = XGBClassifier(n_estimators=200, learning_rate=0.1, max_depth=5,
                     scale_pos_weight=ratio, use_label_encoder=False, eval_metric='logloss', random_state=SEED, n_jobs=-1)

# Voting Classifier (Ensemble)
voting_clf = VotingClassifier(
    estimators=[('lr', clf1), ('rf', clf2), ('xgb', clf3)],
    voting='soft' # Nécessaire pour l'ajustement de seuil plus tard
)

# Création du Pipeline complet : Préprocessing -> SMOTE -> Modèle
# On utilise ImbPipeline pour que SMOTE ne soit appliqué que pendant le .fit() (train), pas le .predict() (test)
binary_pipeline = ImbPipeline([
    ('preprocessor', preprocessor),
    ('smote', SMOTE(random_state=SEED)), # GESTION DÉSÉQUILIBRE V2
    ('classifier', voting_clf)
])

# AMÉLIORATION #1 : Exemple d'Optimisation des Hyperparamètres (GridSearch)
# (Commenté pour gagner du temps d'exécution, mais voici comment faire)
# param_grid = {
#    'classifier__xgb__max_depth': [3, 5],
#    'classifier__rf__n_estimators': [100, 200]
# }
# grid_search = GridSearchCV(binary_pipeline, param_grid, cv=3, scoring='f1', n_jobs=-1)
# grid_search.fit(X_train, y_train_bin)
# best_model_bin = grid_search.best_estimator_
# print(f"Meilleurs paramètres : {grid_search.best_params_}")

print("Entraînement du modèle binaire (Voting + SMOTE)...")
# On entraîne le pipeline directement (sans GridSearch pour cet exemple)
binary_pipeline.fit(X_train, y_train_bin)

# ===================================================
# 5. AMÉLIORATION #3 : Ajustement du Seuil de Décision
# ===================================================
print("Optimisation du seuil de décision...")
# Obtenir les probabilités sur le jeu de test
y_probas = binary_pipeline.predict_proba(X_test)[:, 1]

# Trouver le meilleur seuil pour le F1-score
precisions, recalls, thresholds = precision_recall_curve(y_test_bin, y_probas)
f1_scores = 2 * (precisions * recalls) / (precisions + recalls)
# On enlève la dernière valeur (souvent NaN ou 0)
f1_scores = f1_scores[:-1]
thresholds = thresholds

# Trouver l'index du meilleur F1-score
best_idx = np.argmax(f1_scores)
best_threshold = thresholds[best_idx]
best_f1 = f1_scores[best_idx]

print(f"Seuil optimal trouvé : {best_threshold:.4f}")
print(f"F1-Score maximal théorique sur test : {best_f1:.4f}")

# Appliquer le seuil optimal pour les prédictions finales
y_pred_bin_opt = (y_probas >= best_threshold).astype(int)

print("\n--- Résultats Tâche 1 (Seuil Optimisé) ---")
print(classification_report(y_test_bin, y_pred_bin_opt))
cm = confusion_matrix(y_test_bin, y_pred_bin_opt)
print(f"Matrice de confusion :\n{cm}")
print(f"Cas graves manqués (FN) : {cm[1, 0]} sur {cm[1,0]+cm[1,1]}")


print("\n--- TÂCHE 2 : CLASSIFICATION MULTICLASSE (1-11) ---")
# ===================================================
# 6. TÂCHE 2 : XGBoost (Natif Multiclasse) + Feature Eng.
# ===================================================
# XGBoost gère nativement le multiclasse, souvent mieux que OneVsRest
xgb_multi = XGBClassifier(objective='multi:softmax', num_class=12, # Car classes vont jusqu'à 11
                          n_estimators=200, max_depth=6, learning_rate=0.1,
                          use_label_encoder=False, eval_metric='mlogloss', random_state=SEED, n_jobs=-1)

# Pipeline simple sans SMOTE (souvent moins efficace en multiclasse complexe)
# On pourrait utiliser class_weight='balanced' dans un RandomForest à la place.
multi_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', xgb_multi)
])

# Il faut re-mapper les labels pour XGBoost (doivent commencer à 0)
# Le mappage inverse sera fait après la prédiction
unique_classes = sorted(data['PLT'].unique())
class_map = {cls: i for i, cls in enumerate(unique_classes)}
inv_class_map = {i: cls for cls, i in class_map.items()}

y_train_multi_mapped = y_train_multi.map(class_map)
y_test_multi_mapped = y_test_multi.map(class_map)

print("Entraînement du modèle multiclasse (XGBoost)...")
multi_pipeline.fit(X_train, y_train_multi_mapped)

# Prédictions
y_pred_multi_mapped = multi_pipeline.predict(X_test)
# Remettre les labels d'origine
y_pred_multi = pd.Series(y_pred_multi_mapped).map(inv_class_map)

print("\n--- Résultats Tâche 2 (XGBoost) ---")
print(classification_report(y_test_multi, y_pred_multi, zero_division=0))


--- DÉMARRAGE DE LA VERSION ULTIMATE ---
Chargement des données...
Nettoyage du texte...
Division Train/Test...
--- TÂCHE 1 : CLASSIFICATION BINAIRE (Grave/Non Grave) ---
Entraînement du modèle binaire (Voting + SMOTE)...


/usr/local/lib/python3.12/dist-packages/xgboost/training.py:183: UserWarning: [20:38:39] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


Optimisation du seuil de décision...
Seuil optimal trouvé : 0.6588
F1-Score maximal théorique sur test : 0.8019

--- Résultats Tâche 1 (Seuil Optimisé) ---
              precision    recall  f1-score   support

           0       0.95      0.97      0.96      3850
           1       0.85      0.76      0.80       779

    accuracy                           0.94      4629
   macro avg       0.90      0.87      0.88      4629
weighted avg       0.94      0.94      0.94      4629

Matrice de confusion :
[[3746  104]
 [ 188  591]]
Cas graves manqués (FN) : 188 sur 779

--- TÂCHE 2 : CLASSIFICATION MULTICLASSE (1-11) ---
Entraînement du modèle multiclasse (XGBoost)...


/usr/local/lib/python3.12/dist-packages/xgboost/training.py:183: UserWarning: [20:39:09] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)



--- Résultats Tâche 2 (XGBoost) ---
              precision    recall  f1-score   support

           1       0.83      0.93      0.87      2253
           2       0.84      0.71      0.77       151
           3       0.54      0.30      0.39       194
           4       0.77      0.73      0.75       613
           5       0.64      0.56      0.60       166
           6       0.83      0.80      0.81       193
           7       0.00      0.00      0.00         2
           8       0.76      0.73      0.74       589
           9       1.00      0.75      0.86         4
          10       0.65      0.63      0.64       148
          11       0.64      0.52      0.57       316

    accuracy                           0.79      4629
   macro avg       0.68      0.60      0.64      4629
weighted avg       0.78      0.79      0.78      4629



In [ ]:
# --- Charger et afficher le DataFrame généré ---
output_filename = 'predictions_test_set.csv'

try:
    df_predictions_final = pd.read_csv(output_filename)

    print(f"\nFichier '{output_filename}' chargé avec succès.")
    print(f"Il contient {len(df_predictions_final)} lignes.")

    # (Optionnel) Afficher un aperçu du fichier généré
    print("\nAperçu du fichier de prédictions :")
    print(df_predictions_final.head())

except FileNotFoundError:
    print(f"Erreur: Le fichier '{output_filename}' n'a pas été trouvé. Assurez-vous qu'il a été généré correctement par la fonction précédente.")
except Exception as e:
    print(f"Une erreur est survenue lors du chargement du fichier : {e}")


Fichier 'predictions_test_set.csv' chargé avec succès.
Il contient 4629 lignes.

Aperçu du fichier de prédictions :
   Statut_Grave  Classe_Predite
0             0               1
1             0               1
2             0               6
3             0               1
4             0               1
